# Hybrid quantum machine learning

A **variational quantum classifier** is a hybrid model:

1. a feature map writes the data onto a few qubits
2. a parameterised circuit (the "weights") processes that state
3. the two-qubit parity $\langle ZZ\rangle$ is read out as a score
4. a *classical* optimiser updates the weights

The data is the XOR table — linearly inseparable, so a single classical
perceptron fails. XOR *is* parity, which is why $ZZ$ is the natural
observable. This notebook does not import the QAOA or TSP code.

In [ ]:
import numpy as np
import qiskit as qk
from scipy.optimize import minimize

X = np.array([[0.0, 0.0], [0.0, 1.0], [1.0, 0.0], [1.0, 1.0]])
Y = np.array([0, 1, 1, 0])
PARITY = qk.quantum_info.SparsePauliOp("ZZ")
list(zip(map(tuple, X), Y))

## ZZ feature map and a short RY–CX block

Hadamards plus data phases, then an entangling product phase so the
circuit can see $x_0 x_1$. The trainable block is three CX-linked
layers of $RY$ (8 angles).

In [ ]:
def encode(x):
    qc = qk.QuantumCircuit(2)
    qc.h([0, 1])
    qc.rz(np.pi * x[0], 0)
    qc.rz(np.pi * x[1], 1)
    qc.cx(0, 1)
    qc.rz(np.pi * x[0] * x[1], 1)
    qc.cx(0, 1)
    return qc


def weights(theta):
    qc = qk.QuantumCircuit(2)
    qc.ry(theta[0], 0)
    qc.ry(theta[1], 1)
    qc.cx(0, 1)
    qc.ry(theta[2], 0)
    qc.ry(theta[3], 1)
    qc.cx(1, 0)
    qc.ry(theta[4], 0)
    qc.ry(theta[5], 1)
    qc.cx(0, 1)
    qc.ry(theta[6], 0)
    qc.ry(theta[7], 1)
    return qc


def model(x, theta):
    qc = qk.QuantumCircuit(2)
    qc.compose(encode(x), inplace=True)
    qc.compose(weights(theta), inplace=True)
    return qc


print(model([1, 0], np.zeros(8)).draw())

## Score, loss, train

$\langle ZZ\rangle \in [-1,1]$ is mapped to a $[0,1]$ score (high means
the bits differ). Mean squared error against the XOR labels is handed
to COBYLA.

In [ ]:
def score(x, theta):
    zz = qk.quantum_info.Statevector.from_instruction(model(x, theta)).expectation_value(PARITY)
    return 0.5 * (1.0 - float(np.real(zz)))


def loss(theta):
    return float(np.mean([(score(x, theta) - y) ** 2 for x, y in zip(X, Y)]))


def acc(theta):
    return float(np.mean([int(score(x, theta) >= 0.5) == y for x, y in zip(X, Y)]))


rng = np.random.default_rng(4)
start = rng.uniform(0, 2 * np.pi, size=8)
print("before", loss(start), acc(start))
fit = minimize(loss, start, method="COBYLA", options={"maxiter": 200, "rhobeg": 0.7})
print("after ", loss(fit.x), acc(fit.x), "nfev", fit.nfev)
for x, y in zip(X, Y):
    s = score(x, fit.x)
    print(tuple(x), "score", round(s, 3), "pred", int(s >= 0.5), "label", y)